# 03B — Calibration interne SIMCA à huit tracks

Ce notebook applique les tâches 17–24 du protocole. Les batches 1–2 sont la
seule source de calibration. Un fit train-only alimente les projections objet
et pixel ; les décisions directes utilisent la marge SIMCA et la calibration
3-way est croisée. Les métriques de fragments restent réservées au batch 4.


## 1 — Gouvernance, version et artefacts verrouillés


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError("Launch 03B from the repository or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import (
    build_simca_track_contracts,
    sha256_dataframe,
    verify_frozen_protocol,
)
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import save_parquet
from src.workflows.matrix_preprocessing import (
    assert_wavelength_lock,
    build_wavelength_config,
)
from src.workflows.pca_selection import (
    hash_pca_input_artifacts,
    hash_pca_review_table,
    pca_input_fingerprint,
    validate_pca_preprocessing_shortlist,
)
from src.workflows.protocol_audit import assert_no_forbidden_score_columns
from src.workflows.protocol_split import eligible_object_ids
from src.workflows.simca_internal_calibration import (
    build_calibration_domain_8tracks,
    build_calibration_folds,
    build_internal_calibrated_hyperparameters_8tracks,
    build_internal_calibration_checkpoint_manifest,
    build_internal_calibration_configurations,
    build_reference_object_table,
    expand_projection_configurations,
    load_selected_oof_predictions_from_checkpoint_8tracks,
    resolve_internal_calibration_checkpoint_run_8tracks,
    run_internal_calibration_8tracks,
    summarize_internal_calibration_checkpoint_8tracks,
    validate_internal_calibration_checkpoint_manifest,
)

results_tag = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
protocol_dir = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
qc_dir = PROJECT_ROOT.joinpath(*expcfg.QC_RESULTS_RELATIVE_DIR)
matrix_dir = PROJECT_ROOT / "results" / f"{expcfg.MATRIX_RESULTS_DIR_PREFIX}_{results_tag}"
pca_dir = PROJECT_ROOT / "results" / f"{expcfg.PCA_RESULTS_DIR_PREFIX}_{results_tag}"
output_dir = PROJECT_ROOT / "results" / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = (
    output_dir / expcfg.INTERNAL_CALIBRATION_CHECKPOINT_DIRNAME
    if expcfg.INTERNAL_CALIBRATION_CHECKPOINT_ENABLED else None
)
output_paths = {
    key: output_dir / filename
    for key, filename in expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES.items()
}

verify_frozen_protocol(protocol_dir, strict=True)
protocol_lock = json.loads(
    (protocol_dir / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(
        encoding="utf-8"
    )
)
protocol_hash = str(protocol_lock["lock_sha256"])


In [2]:
split_manifest = pd.read_parquet(
    qc_dir / expcfg.QC_OUTPUT_FILENAMES["split_manifest"]
)
wavelength_lock = pd.read_parquet(
    matrix_dir / expcfg.MATRIX_OUTPUT_FILENAMES["wavelength_config"]
)
pca_selected = pd.read_parquet(
    pca_dir / expcfg.PCA_OUTPUT_FILENAMES["selected"]
)
pca_review = pd.read_parquet(
    pca_dir / expcfg.PCA_OUTPUT_FILENAMES["artifact_review"]
)
pca_input_hash = pca_input_fingerprint(
    hash_pca_input_artifacts(PROJECT_ROOT, results_tag=results_tag)
)
pca_review_hash = hash_pca_review_table(pca_review)
validate_pca_preprocessing_shortlist(
    pca_selected,
    max_per_family=expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY,
    expected_families=expcfg.PCA_SELECTION_EXPECTED_FAMILIES,
    expected_protocol_hash=protocol_hash,
    expected_input_fingerprint=pca_input_hash,
    expected_review_hash=pca_review_hash,
)
if pca_selected["shortlist_id"].astype(str).nunique() != 1:
    raise RuntimeError("03B requires one locked PCA shortlist_id.")
pca_shortlist_id = str(pca_selected["shortlist_id"].iloc[0])

object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=True,
)
if expcfg.USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(next(iter(object_db.values()))["wavelengths"])
wavelength_candidate = build_wavelength_config(
    image_db,
    object_db,
    wavelength_mode=expcfg.DEFAULT_WAVELENGTH_MODE,
    protocol_version=expcfg.PROTOCOL_VERSION,
    n_remove_start=expcfg.N_REMOVE_START,
    n_stop_end=expcfg.N_STOP_END,
    window_min_nm=(
        expcfg.WAVELENGTH_WINDOW_MIN_NM
        if expcfg.USE_WAVELENGTH_WINDOW else None
    ),
    window_max_nm=(
        expcfg.WAVELENGTH_WINDOW_MAX_NM
        if expcfg.USE_WAVELENGTH_WINDOW else None
    ),
)
assert_wavelength_lock(wavelength_lock, wavelength_candidate)


## 2 — Contrats des huit tracks


In [3]:
track_contracts = build_simca_track_contracts()
if len(track_contracts) != 8 or set(track_contracts["track_id"]) != {
    f"E{index}" for index in range(1, 9)
}:
    raise RuntimeError("The materialised contract must contain exactly E1-E8.")
save_parquet(track_contracts, output_paths["track_contracts"])
track_contract_hash = sha256_dataframe(track_contracts)
display(track_contracts)


,track_id,evaluation_track,parent_track,training_matrix_family,projection_level,projection_matrix_policy,allowed_projection_methods_json,primary_unit,decision_mode,decision_score_type,higher_is_target,direct_2way_threshold,constraint_profile_id,calibration_primary_metrics_json,final_evaluation_metrics_json,pareto_minimize_json,pareto_maximize_json,protocol_version,schema_version
0,E1,object_train__object_projection__2way,object_matrix_2way,object_matrix,object_projection,match_object_training_method,"[""object_mean"",""object_median""]",object,2way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""target_miss_rate"",""false_accept_rate"",""balan...",[],"[""target_miss_rate"",""false_accept_rate""]","[""balanced_accuracy""]",8tracks_v3,8tracks_v2
1,E2,object_train__object_projection__3way,object_matrix_3way,object_matrix,object_projection,match_object_training_method,"[""object_mean"",""object_median""]",object,3way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""target_miss_rate"",""false_accept_rate"",""uncer...",[],"[""target_miss_rate"",""false_accept_rate"",""uncer...","[""coverage_rate"",""decided_balanced_accuracy""]",8tracks_v3,8tracks_v2
2,E3,object_train__pixel_projection__2way,object_matrix_2way,object_matrix,pixel_projection,all_pixels,"[""all_pixels""]",source_image,2way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""macro_image_target_miss_rate"",""macro_image_f...","[""small_fragment_recall"",""fragment_precision""]","[""macro_image_target_miss_rate"",""macro_image_f...","[""macro_image_balanced_accuracy""]",8tracks_v3,8tracks_v2
3,E4,object_train__pixel_projection__3way,object_matrix_3way,object_matrix,pixel_projection,all_pixels,"[""all_pixels""]",source_image,3way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""macro_image_target_miss_rate"",""macro_image_f...","[""small_fragment_recall"",""fragment_precision""]","[""macro_image_target_miss_rate"",""macro_image_f...","[""coverage_rate"",""decided_balanced_accuracy""]",8tracks_v3,8tracks_v2
4,E5,pixel_train__object_projection__2way,pixel_matrix_2way,pixel_matrix,object_projection,compare_object_mean_and_median,"[""object_mean"",""object_median""]",object,2way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""target_miss_rate"",""false_accept_rate"",""balan...",[],"[""target_miss_rate"",""false_accept_rate""]","[""balanced_accuracy""]",8tracks_v3,8tracks_v2
5,E6,pixel_train__object_projection__3way,pixel_matrix_3way,pixel_matrix,object_projection,compare_object_mean_and_median,"[""object_mean"",""object_median""]",object,3way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""target_miss_rate"",""false_accept_rate"",""uncer...",[],"[""target_miss_rate"",""false_accept_rate"",""uncer...","[""coverage_rate"",""decided_balanced_accuracy""]",8tracks_v3,8tracks_v2
6,E7,pixel_train__pixel_projection__2way,pixel_matrix_2way,pixel_matrix,pixel_projection,all_pixels,"[""all_pixels""]",source_image,2way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""macro_image_target_miss_rate"",""macro_image_f...","[""small_fragment_recall"",""fragment_precision""]","[""macro_image_target_miss_rate"",""macro_image_f...","[""macro_image_balanced_accuracy""]",8tracks_v3,8tracks_v2
7,E8,pixel_train__pixel_projection__3way,pixel_matrix_3way,pixel_matrix,pixel_projection,all_pixels,"[""all_pixels""]",source_image,3way,simca_margin,True,0.0,internal_calibration_risk_profile,"[""macro_image_target_miss_rate"",""macro_image_f...","[""small_fragment_recall"",""fragment_precision""]","[""macro_image_target_miss_rate"",""macro_image_f...","[""coverage_rate"",""decided_balanced_accuracy""]",8tracks_v3,8tracks_v2


## 3 — Références QC-éligibles et folds groupés


In [4]:
calibration_ids = eligible_object_ids(split_manifest, "calibration")
reference_objects = build_reference_object_table(
    object_db,
    allowed_object_ids=calibration_ids,
    batches=expcfg.INTERNAL_CALIBRATION_BATCHES,
    classes=expcfg.REFERENCE_CLASSES,
)
if set(reference_objects["batch"]).intersection(
    expcfg.INTERNAL_CALIBRATION_FORBIDDEN_BATCHES
):
    raise RuntimeError("A forbidden batch entered the 03B reference table.")
calibration_folds, fold_diagnostics = build_calibration_folds(
    reference_objects,
    n_splits=expcfg.INTERNAL_CALIBRATION_N_SPLITS,
    n_size_bins=expcfg.INTERNAL_CALIBRATION_SIZE_N_BINS,
    random_state=expcfg.INTERNAL_CALIBRATION_FOLD_RANDOM_STATE,
    require_complete_coverage=True,
)
if set(calibration_folds["object_id"].astype(str)) - set(calibration_ids):
    raise RuntimeError("An object outside the QC calibration split entered 03B.")
save_parquet(calibration_folds, output_paths["folds"])
save_parquet(fold_diagnostics, output_paths["fold_diagnostics"])
fold_contract_hash = sha256_dataframe(calibration_folds)
display(fold_diagnostics)


,fold_id,n_groups,n_objects,n_target_objects,n_non_target_objects,n_batch_1_objects,n_batch_2_objects,median_object_size,coverage_complete
0,0,2,104,52,52,52,52,75.5,True
1,1,2,105,46,59,46,59,65.0,True


## 4 — Domaine des fits et des projections


In [5]:
fit_configurations = build_internal_calibration_configurations(
    pca_selected,
    matrix_methods=expcfg.INTERNAL_CALIBRATION_MATRIX_METHODS,
    m_values=expcfg.INTERNAL_CALIBRATION_M_VALUES,
    pixel_strategies=expcfg.INTERNAL_CALIBRATION_PIXEL_STRATEGIES,
    n_components_values=expcfg.INTERNAL_CALIBRATION_N_COMPONENTS_VALUES,
    rule_variants=expcfg.INTERNAL_CALIBRATION_RULE_VARIANTS,
    alpha_values=expcfg.INTERNAL_CALIBRATION_ALPHA_VALUES,
    sg_windows=expcfg.INTERNAL_CALIBRATION_SG_WINDOWS,
    sg_polyorders=expcfg.INTERNAL_CALIBRATION_SG_POLYORDERS,
    dilation_radii=expcfg.INTERNAL_CALIBRATION_DILATION_RADII,
    random_seeds=expcfg.INTERNAL_CALIBRATION_RANDOM_SEEDS,
    max_configs=expcfg.INTERNAL_CALIBRATION_MAX_CONFIGS,
)
evaluation_configurations = expand_projection_configurations(
    fit_configurations,
    track_contracts,
)
evaluation_configurations["protocol_hash"] = protocol_hash
evaluation_configurations["pca_shortlist_id"] = pca_shortlist_id
configuration_hash = sha256_dataframe(evaluation_configurations)
display(
    evaluation_configurations.groupby(
        ["track_id", "evaluation_track", "projection_matrix_method"],
        as_index=False,
    ).agg(
        n_evaluations=("evaluation_config_id", "nunique"),
        n_fits=("fit_config_id", "nunique"),
    )
)


,track_id,evaluation_track,projection_matrix_method,n_evaluations,n_fits
0,E1,object_train__object_projection__2way,object_mean,3840,480
1,E1,object_train__object_projection__2way,object_median,3840,480
2,E2,object_train__object_projection__3way,object_mean,3840,480
3,E2,object_train__object_projection__3way,object_median,3840,480
4,E3,object_train__pixel_projection__2way,all_pixels,7680,960
5,E4,object_train__pixel_projection__3way,all_pixels,7680,960
6,E5,pixel_train__object_projection__2way,object_mean,15360,1920
7,E5,pixel_train__object_projection__2way,object_median,15360,1920
8,E6,pixel_train__object_projection__3way,object_mean,15360,1920
9,E6,pixel_train__object_projection__3way,object_median,15360,1920


## 5 — Exécution OOF unique avec fits partagés

Les objets proviennent exclusivement du split QC accepté. La validité qui
dépend du prétraitement est ensuite appliquée aux projections : une ligne
pixel contenant une réflectance non positive est exclue pour `absorbance`,
jamais clippée. Les nombres initial, conservé et exclu sont ajoutés à l'audit.
Une projection techniquement invalide est journalisée par configuration sans
interrompre les autres fits. Les prédictions candidates restent dans les
shards de checkpoint et les métriques sont réduites un shard à la fois afin
que la consommation de RAM ne croisse pas avec le nombre de configurations.


In [ ]:
if not expcfg.INTERNAL_CALIBRATION_RUN:
    raise RuntimeError("Enable INTERNAL_CALIBRATION_RUN in experiment_config.py.")
checkpoint_state = run_internal_calibration_8tracks(
    object_db=object_db,
    folds=calibration_folds,
    configurations=evaluation_configurations,
    wavelengths=wavelengths,
    under_m_policy=expcfg.INTERNAL_CALIBRATION_UNDER_M_POLICY,
    verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
    checkpoint_dir=checkpoint_dir,
    checkpoint_context={
        "protocol_hash": protocol_hash,
        "pca_shortlist_id": pca_shortlist_id,
        "track_contract_hash": track_contract_hash,
        "fold_contract_hash": fold_contract_hash,
        "configuration_hash": configuration_hash,
    },
    resume_from_checkpoint=expcfg.INTERNAL_CALIBRATION_RESUME_FROM_CHECKPOINT,
    materialize_checkpoint_results=False,
)
calibration_results = summarize_internal_calibration_checkpoint_8tracks(
    checkpoint_state["checkpoint_run_dir"],
    evaluation_configurations,
    verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
)
fit_diagnostics = calibration_results["fit_diagnostics"]
rule_diagnostics = calibration_results["rule_diagnostics"]
projection_shift = calibration_results["projection_shift"]
technical_errors = calibration_results["technical_errors"]
metrics_2way = calibration_results["metrics_2way"]
pixel_votes_2way = calibration_results["pixel_votes_2way"]
thresholds_3way = calibration_results["thresholds_3way"]
thresholds_3way_study = calibration_results["thresholds_3way_study"]
for table, key in (
    (fit_diagnostics, "fit_diagnostics"),
    (rule_diagnostics, "rule_diagnostics"),
    (projection_shift, "projection_shift"),
):
    save_parquet(table, output_paths[key])


## 6 — Décisions 2-way directes et agrégations secondaires


In [ ]:
# Metrics already reduced shard by shard during stage 5.
save_parquet(metrics_2way, output_paths["oof_2way_metrics"])
save_parquet(
    pixel_votes_2way,
    output_paths["pixel_to_object_thresholds_2way"],
)


## 7 — Calibration croisée des seuils 3-way


In [ ]:
# Three-way thresholds already reduced shard by shard during stage 5.
if not thresholds_3way.loc[
    thresholds_3way["score_type"].eq("simca_margin"),
    "three_way_lower_threshold",
].dropna().lt(0.0).all():
    raise RuntimeError("Direct lower 3-way margins must be negative.")
if not thresholds_3way.loc[
    thresholds_3way["score_type"].eq("simca_margin"),
    "three_way_upper_threshold",
].dropna().gt(0.0).all():
    raise RuntimeError("Direct upper 3-way margins must be positive.")
save_parquet(thresholds_3way, output_paths["thresholds_3way"])
save_parquet(thresholds_3way_study, output_paths["thresholds_3way_study"])
display(thresholds_3way_study)


## 8 — Sélection k/m/règles/graines et Pareto par track


In [6]:
# This cell can be resumed from the compact artifacts without rerunning stage 5.
if "metrics_2way" not in globals():
    metrics_2way = pd.read_parquet(output_paths["oof_2way_metrics"])
if "thresholds_3way" not in globals():
    thresholds_3way = pd.read_parquet(output_paths["thresholds_3way"])
if "technical_errors" not in globals():
    if output_paths["calibration_audit"].exists():
        previous_audit = pd.read_parquet(output_paths["calibration_audit"])
        technical_errors = previous_audit.loc[
            ~previous_audit["audit_type"].eq("selection_funnel")
        ].reindex(columns=expcfg.INTERNAL_CALIBRATION_AUDIT_V2_COLUMNS)
    else:
        technical_errors = pd.DataFrame(
            columns=expcfg.INTERNAL_CALIBRATION_AUDIT_V2_COLUMNS
        )

declared_unsupported_track_ids = {"E3"}
calibrated_hyperparameters, calibration_audit = (
    build_internal_calibrated_hyperparameters_8tracks(
        evaluation_configurations,
        metrics_2way,
        thresholds_3way,
        tolerance=expcfg.INTERNAL_CALIBRATION_PERFORMANCE_PLATEAU_TOLERANCE,
        allowed_unsupported_track_ids=declared_unsupported_track_ids,
    )
)
if not technical_errors.empty:
    calibration_audit = pd.concat(
        [calibration_audit, technical_errors],
        ignore_index=True,
        sort=False,
    ).reindex(columns=expcfg.INTERNAL_CALIBRATION_AUDIT_V2_COLUMNS)
save_parquet(
    calibrated_hyperparameters,
    output_paths["calibrated_hyperparameters"],
)
save_parquet(calibration_audit, output_paths["calibration_audit"])
selection_audit = calibration_audit.loc[
    calibration_audit["audit_type"].eq("selection_funnel")
].copy()
display(selection_audit)
observed_unsupported_track_ids = set(
    selection_audit.loc[
        selection_audit["track_status"].eq("unsupported"), "track_id"
    ].astype(str)
)
if observed_unsupported_track_ids != declared_unsupported_track_ids:
    raise RuntimeError(
        "The unsupported-track declaration does not match the audit: "
        f"declared={sorted(declared_unsupported_track_ids)}, "
        f"observed={sorted(observed_unsupported_track_ids)}"
    )
failed_tracks = selection_audit.loc[
    ~selection_audit["track_status"].isin({"calibrated", "unsupported"})
]
if not failed_tracks.empty:
    raise RuntimeError(
        "An undeclared track failed calibration; inspect calibration_audit.parquet."
    )
print("Explicit unsupported internal-calibration tracks:", sorted(observed_unsupported_track_ids))


,audit_type,evaluation_track,track_id,n_initial,n_technical_valid,n_risk_feasible,n_k_plateau,n_m_plateau,n_seed_consensus,n_pareto,track_status,failure_reason
0,selection_funnel,object_train__object_projection__2way,E1,7680,7680,3125.0,690.0,690.0,690.0,10.0,calibrated,
1,selection_funnel,object_train__object_projection__3way,E2,7680,7673,70762.0,70762.0,70762.0,70762.0,402.0,calibrated,
2,selection_funnel,object_train__pixel_projection__2way,E3,7680,7680,0.0,0.0,0.0,0.0,0.0,unsupported,risk_constraints
3,selection_funnel,object_train__pixel_projection__3way,E4,7680,2931,17256.0,17256.0,17256.0,17256.0,220.0,calibrated,
4,selection_funnel,pixel_train__object_projection__2way,E5,30720,30720,3839.0,546.0,367.0,125.0,1.0,calibrated,
5,selection_funnel,pixel_train__object_projection__3way,E6,30720,19278,87628.0,87628.0,87628.0,22796.0,10.0,calibrated,
6,selection_funnel,pixel_train__pixel_projection__2way,E7,15360,15360,2133.0,287.0,206.0,76.0,7.0,calibrated,
7,selection_funnel,pixel_train__pixel_projection__3way,E8,15360,11415,150212.0,150212.0,150212.0,36744.0,397.0,calibrated,


Explicit unsupported internal-calibration tracks: ['E3']


In [7]:
selection_audit.loc[selection_audit["track_status"].eq("unsupported")]

,audit_type,evaluation_track,track_id,n_initial,n_technical_valid,n_risk_feasible,n_k_plateau,n_m_plateau,n_seed_consensus,n_pareto,track_status,failure_reason
2,selection_funnel,object_train__pixel_projection__2way,E3,7680,7680,0.0,0.0,0.0,0.0,0.0,unsupported,risk_constraints


## 9 — Domaine final, intégrité et sauvegarde


In [8]:
calibration_domain = build_calibration_domain_8tracks(
    calibrated_hyperparameters,
    evaluation_configurations,
    pca_shortlist_id=pca_shortlist_id,
    protocol_hash=protocol_hash,
    unsupported_track_ids=observed_unsupported_track_ids,
)
save_parquet(calibration_domain, output_paths["calibration_domain"])
checkpoint_run_dir = resolve_internal_calibration_checkpoint_run_8tracks(
    checkpoint_dir,
    checkpoint_context={
        "protocol_hash": protocol_hash,
        "pca_shortlist_id": pca_shortlist_id,
        "track_contract_hash": track_contract_hash,
        "fold_contract_hash": fold_contract_hash,
        "configuration_hash": configuration_hash,
    },
    expected_fit_config_ids=evaluation_configurations[
        "fit_config_id"
    ].astype(str).unique(),
)
oof_objects, oof_pixels = (
    load_selected_oof_predictions_from_checkpoint_8tracks(
        checkpoint_run_dir,
        calibration_domain,
    )
)
save_parquet(oof_objects, output_paths["oof_object_predictions"])
save_parquet(oof_pixels, output_paths["oof_pixel_predictions"])

assert_no_forbidden_score_columns(
    {
        "metrics_2way": metrics_2way,
        "thresholds_3way": thresholds_3way,
        "calibrated_hyperparameters": calibrated_hyperparameters,
        "calibration_domain": calibration_domain,
    }
)
checkpoint_manifest = build_internal_calibration_checkpoint_manifest(
    output_paths,
    checkpoint_dir=checkpoint_dir,
    protocol_hash=protocol_hash,
    pca_shortlist_id=pca_shortlist_id,
    track_contract_hash=track_contract_hash,
    fold_contract_hash=fold_contract_hash,
    configuration_hash=configuration_hash,
    completed_fit_config_ids=evaluation_configurations[
        "fit_config_id"
    ].astype(str).unique(),
)
output_paths["checkpoint_manifest"].write_text(
    json.dumps(checkpoint_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
validate_internal_calibration_checkpoint_manifest(
    checkpoint_manifest,
    expected_fit_config_ids=evaluation_configurations[
        "fit_config_id"
    ].astype(str).unique(),
)
display(
    pd.DataFrame(checkpoint_manifest["shards"])[
        ["name", "row_count", "file_sha256"]
    ]
)
print(
    f"03B {expcfg.PROTOCOL_VERSION} completed; "
    "03C must audit calibration_domain.parquet before 04A/04B."
)


,name,row_count,file_sha256
0,calibrated_hyperparameters,1047,f1a8d5d4041da5d92eb8ddee6efc36eff1d2660bfc8aca...
1,calibration_audit,116,731175601f1a92247fbd2e708d185d429c9debae45fc1c...
2,calibration_domain,1059,0d40601938d0781a908f17e0c449bb206acef7c4449aad...
3,fit_diagnostics,5760,575e7e903dca2b394425315bc5db7b4b8803036aa90385...
4,fold_diagnostics,2,7a9f81b246d590315eea52c21abdec908450501f059510...
...,...,...,...
1504,rule_diagnostics,320,1d3956deb9590e4412ce676b10f2a6da6e921adbc4791a...
1505,oof_object_predictions,16720,896cbd531057cea144a0ab5e5f1911cab656f388dbe532...
1506,oof_pixel_predictions,1195600,be7281bc04cbc569b966b5f7c6a72ade6a316397ffa7cd...
1507,projection_shift,320,f7f6ee45b45aae6b9669db077c71eea673988641a29012...


03B 8tracks_v3 completed; 03C must audit calibration_domain.parquet before 04A/04B.
